# Run Multiple Local LLMs on One Server with Llama-Swap

## Beginner-Friendly End-to-End Guide

This notebook teaches how to run multiple local Large Language Models (LLMs) on one machine using **Llama-Swap**.

The goal is simple:

> Use one local API endpoint and switch between different local LLMs automatically.

Instead of manually starting and stopping different model servers, Llama-Swap works like a smart proxy. You send a request with a model name, and Llama-Swap starts the correct model server for you.

---

## What you will learn

By the end of this notebook, you will understand:

1. What problem Llama-Swap solves
2. How local LLM serving works
3. What `llama.cpp`, `llama-server`, GGUF models, and OpenAI-compatible APIs mean
4. How to install the required tools
5. How to download small local models from Hugging Face
6. How to create a Llama-Swap configuration file
7. How to run Llama-Swap
8. How to send test requests to different models
9. How to troubleshoot common errors
10. How to extend this setup for RAG, LangChain, FastAPI, and AI agents

---

## Who is this notebook for?

This guide is for data science learners, AI/ML students, ML engineers, AI engineers, people learning local LLM infrastructure, and anyone who wants to reduce API costs and experiment privately.

You do **not** need advanced DevOps knowledge to follow this.


# 1. The Problem: Running Multiple Local LLMs Is Usually Messy

Suppose you want to use different local LLMs for different tasks:

| Task | Possible Model |
|---|---|
| Coding help | Qwen / DeepSeek Coder |
| Simple chat | SmolLM / TinyLlama |
| Writing | Mistral / Llama |
| Fast lightweight testing | Small GGUF model |
| RAG system | A model optimized for instruction following |

Normally, each local model server needs its own port, terminal, command, manual start/stop, and memory management.

Example problem:

```text
Model A running on port 8001
Model B running on port 8002
Model C running on port 8003
Your app now needs to remember which model is on which port.
```

This is not clean.

---

## What we want instead

We want one endpoint:

```text
http://localhost:8080
```

Then we want to switch models just by changing the `"model"` field:

```json
{
  "model": "qwen2.5",
  "prompt": "Explain Python simply"
}
```

This is what Llama-Swap helps us do.


# 2. What Is Llama-Swap?

**Llama-Swap** is a lightweight proxy server for local LLMs.

Think of it like a traffic controller.

Your application sends a request to Llama-Swap. Llama-Swap checks which model you asked for. Then it starts the correct backend model server and forwards the request.

Without Llama-Swap:

```text
You -> Model A server
You -> Model B server
You -> Model C server
```

With Llama-Swap:

```text
You -> Llama-Swap -> Correct model server
```

## Why this is useful

Llama-Swap can expose one OpenAI-compatible API endpoint, start models only when needed, stop unused models, reduce wasted memory, make local LLM experiments cleaner, and manage several models from one config file.

## Important idea

Llama-Swap does **not** run the LLM by itself.

It manages other model servers.

In this notebook, the actual model server will be:

```text
llama-server
```

which comes from:

```text
llama.cpp
```


# 3. Key Terms You Must Understand

## LLM

A **Large Language Model** is a model that generates text, answers questions, summarizes documents, writes code, and more.

Examples include Llama, Qwen, Mistral, Phi, Gemma, and SmolLM.

## Local LLM

A **local LLM** runs on your own machine instead of using a cloud API.

Benefits:

- better privacy
- no API cost per request
- useful for experimentation
- works offline after setup
- good for learning AI infrastructure

Limitations:

- slower without a strong GPU
- large models need more RAM/VRAM
- setup can be more technical

## GGUF

**GGUF** is a common file format used by `llama.cpp` for running quantized local models.

Example model file:

```text
Qwen2.5-0.5B-Instruct-Q4_K_M.gguf
```

The `.gguf` file is the actual model file you download.

## Quantization

Quantization makes models smaller and easier to run.

Example:

```text
Q4_K_M
```

usually means a 4-bit quantized version.

## llama.cpp

`llama.cpp` is a popular open-source project for running LLMs efficiently on CPUs and GPUs. It provides a server called `llama-server`.

## OpenAI-Compatible API

This means the local server accepts requests that look similar to OpenAI API requests.

Examples:

```text
POST /v1/completions
POST /v1/chat/completions
GET /v1/models
```

This is useful because many tools already know how to talk to OpenAI-style APIs.


# 4. Final Architecture

Here is the setup we are building:

```text
Client / Python / curl / App
          |
          v
http://localhost:8080
          |
          v
     Llama-Swap
          |
          |--- starts qwen2.5 when model = "qwen2.5"
          |
          |--- starts smollm2 when model = "smollm2"
          |
          v
      llama-server
          |
          v
      GGUF model file
```

You only talk to:

```text
http://localhost:8080
```

Llama-Swap handles the rest.


# 5. Tools We Need

| Tool | Why we need it |
|---|---|
| Python 3.8+ | General scripting and Hugging Face CLI |
| Homebrew | Easy installation on macOS |
| llama.cpp / llama-server | Runs the GGUF model files |
| Hugging Face CLI | Downloads model files |
| Llama-Swap | Switches between local model servers |
| jq | Pretty-prints JSON output in terminal |
| curl | Sends API requests |

## OS note

This notebook is written mainly for macOS, especially Apple Silicon M1/M2/M3. The same concepts also work on Linux and Windows. For Windows, WSL2 is usually easier.


# 6. Check Your System

Run these commands to understand your system.

Check Python:


In [ ]:
!python3 --version


Check your machine type:


In [ ]:
!uname -m


Common outputs:

| Output | Meaning |
|---|---|
| `arm64` | Apple Silicon Mac |
| `x86_64` | Intel/AMD machine |
| `aarch64` | ARM Linux machine |

This matters because you need the correct Llama-Swap binary for your system.


# 7. Install Required Tools

## 7.1 Install Homebrew

If you are on macOS and do not have Homebrew, install it from the official Homebrew website.

Check if Homebrew is installed:


In [ ]:
!brew --version


## 7.2 Install llama.cpp

On macOS:

```bash
brew install llama.cpp
```

This should install the `llama-server` command.


In [ ]:
!which llama-server


If the command above returns a path, `llama-server` is installed.

## 7.3 Install Hugging Face CLI

We use the Hugging Face CLI to download models.

Newer versions provide the `hf` command. Older examples may use `huggingface-cli`.

Install it:


In [ ]:
!pip install -U "huggingface_hub[cli]"


Check installation:


In [ ]:
!hf --help


If `hf` does not work, try:

```bash
huggingface-cli --help
```

## 7.4 Install jq

`jq` makes JSON output easier to read.

On macOS:

```bash
brew install jq
```

Check it:


In [ ]:
!jq --version


# 8. Create a Project Folder

Recommended folder structure:

```text
llama-swap-local-llms/
│
├── config.yaml
├── llama-swap
├── models/
│   ├── SmolLM2-135M-Instruct-Q4_K_M.gguf
│   └── Qwen2.5-0.5B-Instruct-Q4_K_M.gguf
└── README.md
```

Create the folders:


In [ ]:
!mkdir -p llama-swap-local-llms/models
!cd llama-swap-local-llms && pwd


# 9. Download Llama-Swap

Llama-Swap is usually distributed as a single binary file.

You should download the correct release for your operating system and CPU architecture from the official GitHub releases page.

## Important: do not blindly copy an old version

Some tutorials hardcode a version such as `v126`. That may become outdated.

Better approach:

1. Open the Llama-Swap GitHub releases page.
2. Find the latest release.
3. Download the correct file for your system.

## Example for Apple Silicon Mac

Your file name will usually include something like:

```text
darwin_arm64
```

Example pattern:

```text
llama-swap_VERSION_darwin_arm64.tar.gz
```

Manual download example:

```bash
cd llama-swap-local-llms

curl -L -o llama-swap.tar.gz \
  https://github.com/mostlygeek/llama-swap/releases/download/<LATEST_VERSION>/<CORRECT_FILE_NAME>.tar.gz

tar -xzf llama-swap.tar.gz
chmod +x llama-swap
./llama-swap --version
```

Replace `<LATEST_VERSION>` and `<CORRECT_FILE_NAME>` with the actual latest release and file name from GitHub.


# 10. Download Two Small Models

For beginners, use small models first.

Large models can be slow and may fail if your machine does not have enough memory.

We will use:

| Model | Size | Why use it |
|---|---:|---|
| SmolLM2-135M-Instruct | Very small | Fast test model |
| Qwen2.5-0.5B-Instruct | Small | Better response quality than tiny models |

Both are downloaded in GGUF format.

## 10.1 Download SmolLM2


In [ ]:
!hf download bartowski/SmolLM2-135M-Instruct-GGUF \
  SmolLM2-135M-Instruct-Q4_K_M.gguf \
  --local-dir llama-swap-local-llms/models


If your system uses the older CLI, use:

```bash
huggingface-cli download bartowski/SmolLM2-135M-Instruct-GGUF \
  --include "SmolLM2-135M-Instruct-Q4_K_M.gguf" \
  --local-dir llama-swap-local-llms/models
```

## 10.2 Download Qwen2.5


In [ ]:
!hf download bartowski/Qwen2.5-0.5B-Instruct-GGUF \
  Qwen2.5-0.5B-Instruct-Q4_K_M.gguf \
  --local-dir llama-swap-local-llms/models


Confirm downloaded files:


In [ ]:
!ls -lh llama-swap-local-llms/models


# 11. Test One Model Directly with llama-server

Before using Llama-Swap, test that `llama-server` works.

This is important because if `llama-server` itself does not work, Llama-Swap will not work either.

## 11.1 Start SmolLM2 manually

Run this in a separate terminal:

```bash
cd llama-swap-local-llms

llama-server \
  --model models/SmolLM2-135M-Instruct-Q4_K_M.gguf \
  --port 8001
```

Keep that terminal running.

## 11.2 Test the server

Open another terminal and run:

```bash
curl -s http://localhost:8001/v1/completions \
  -H "Content-Type: application/json" \
  -d '{
        "prompt": "User: Explain Python in one sentence.\nAssistant:",
        "max_tokens": 60
      }' | jq
```

If you get a text response, your model server works.

## 11.3 Stop the manual server

Press `CTRL + C` in the terminal where `llama-server` is running.


# 12. Create the Llama-Swap Config File

Llama-Swap uses a YAML config file.

This file tells Llama-Swap:

- model names
- how to start each model
- which command to run
- where the model file is
- which port to assign

We will create:

```text
config.yaml
```

Important:

```text
${PORT}
```

is a placeholder. Llama-Swap automatically fills it with a free port.


In [ ]:
from pathlib import Path

project_dir = Path("llama-swap-local-llms").resolve()
models_dir = project_dir / "models"

config_text = f"""
models:
  "smollm2":
    cmd: |
      llama-server
      --model {models_dir / "SmolLM2-135M-Instruct-Q4_K_M.gguf"}
      --port ${{PORT}}

  "qwen2.5":
    cmd: |
      llama-server
      --model {models_dir / "Qwen2.5-0.5B-Instruct-Q4_K_M.gguf"}
      --port ${{PORT}}
"""

config_path = project_dir / "config.yaml"
config_path.write_text(config_text.strip() + "\n")

print(config_path)
print(config_path.read_text())


# 13. Understand the Config File

```yaml
models:
```

This section lists all models Llama-Swap can manage.

```yaml
"smollm2":
```

This is the model name you will use in your API request.

Example:

```json
{
  "model": "smollm2"
}
```

```yaml
cmd: |
```

This is the command Llama-Swap runs when it needs to start that model.

```yaml
llama-server
--model /path/to/model.gguf
--port ${PORT}
```

This starts `llama-server` using a specific GGUF model file.

`${PORT}` means Llama-Swap chooses an available internal port automatically.


# 14. Run Llama-Swap

Now start Llama-Swap.

Run this in a terminal:

```bash
cd llama-swap-local-llms

./llama-swap \
  --config config.yaml \
  --listen 127.0.0.1:8080
```

This starts the Llama-Swap proxy at:

```text
http://127.0.0.1:8080
```

At first, no model may be loaded. The model starts when the first request arrives.


# 15. Test Model Switching

Now send requests to Llama-Swap.

The important part is this field:

```json
"model": "qwen2.5"
```

or:

```json
"model": "smollm2"
```

Llama-Swap uses this field to decide which backend model to start.

## 15.1 Test Qwen2.5


In [ ]:
%%bash
curl -s http://localhost:8080/v1/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer no-key" \
  -d '{
        "model": "qwen2.5",
        "prompt": "User: Explain Python in one sentence.\nAssistant:",
        "max_tokens": 80
      }' | jq '.choices[0].text'


## 15.2 Test SmolLM2


In [ ]:
%%bash
curl -s http://localhost:8080/v1/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer no-key" \
  -d '{
        "model": "smollm2",
        "prompt": "User: Explain Python in one sentence.\nAssistant:",
        "max_tokens": 80
      }' | jq '.choices[0].text'


# 16. What Just Happened?

When you requested `"model": "qwen2.5"`, Llama-Swap checked the config file and started the Qwen model server.

When you then requested `"model": "smollm2"`, Llama-Swap switched to SmolLM2.

By default, a simple Llama-Swap setup is useful for efficient swapping, so it can avoid keeping every model loaded at the same time.

This is helpful when your machine has limited RAM or VRAM.


# 17. Use Chat Completions Instead of Text Completions

Modern LLM applications usually use:

```text
/v1/chat/completions
```

instead of:

```text
/v1/completions
```

The chat format uses messages:

```json
[
  {"role": "system", "content": "You are helpful."},
  {"role": "user", "content": "Explain Python simply."}
]
```

Try this:


In [ ]:
%%bash
curl -s http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer no-key" \
  -d '{
        "model": "qwen2.5",
        "messages": [
          {
            "role": "system",
            "content": "You are a helpful AI tutor. Explain concepts simply."
          },
          {
            "role": "user",
            "content": "Explain what a local LLM is in simple words."
          }
        ],
        "max_tokens": 120
      }' | jq '.choices[0].message.content'


# 18. Use Python to Call Your Local LLM API

You can also call the Llama-Swap endpoint from Python.

This is useful for notebooks, data science workflows, RAG pipelines, AI agents, FastAPI apps, and automation scripts.


In [ ]:
!pip install requests


In [ ]:
import requests

url = "http://localhost:8080/v1/chat/completions"

payload = {
    "model": "smollm2",
    "messages": [
        {
            "role": "system",
            "content": "You are a beginner-friendly AI tutor."
        },
        {
            "role": "user",
            "content": "Explain what Llama-Swap does in 3 bullet points."
        }
    ],
    "max_tokens": 150
}

response = requests.post(url, json=payload)
response.raise_for_status()

data = response.json()
print(data["choices"][0]["message"]["content"])


# 19. Use the OpenAI Python Client with a Local Endpoint

Because the endpoint is OpenAI-compatible, many OpenAI-style clients can work with it.


In [ ]:
!pip install openai


In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="no-key"
)

response = client.chat.completions.create(
    model="qwen2.5",
    messages=[
        {
            "role": "system",
            "content": "You are a clear technical teacher."
        },
        {
            "role": "user",
            "content": "Explain why local LLMs are useful for data scientists."
        }
    ],
    max_tokens=150
)

print(response.choices[0].message.content)


# 20. Optional: Run Multiple Models Together Using Groups

By default, many Llama-Swap workflows focus on swapping models to save memory.

But sometimes you may want small models to run together.

This is where groups are useful.

Example idea:

```yaml
groups:
  "small-models":
    swap: false
    members:
      - smollm2
      - qwen2.5
```

Conceptually:

| Setting | Meaning |
|---|---|
| `swap: true` | Models in the group may be swapped/unloaded |
| `swap: false` | Models in the group can stay running together |

Use this carefully because running multiple models at the same time uses more memory.


# 21. Better Config with Practical Options

A more practical config may include model aliases.

Example:

```yaml
models:
  "smollm2":
    aliases:
      - "small"
      - "fast"
    cmd: |
      llama-server
      --model /absolute/path/to/SmolLM2-135M-Instruct-Q4_K_M.gguf
      --port ${PORT}

  "qwen2.5":
    aliases:
      - "qwen"
      - "balanced"
    cmd: |
      llama-server
      --model /absolute/path/to/Qwen2.5-0.5B-Instruct-Q4_K_M.gguf
      --port ${PORT}
```

Aliases let you call the same model using different names.

Example:

```json
{
  "model": "fast"
}
```

could route to `smollm2`.


# 22. How This Helps in Real AI Projects

## RAG Systems

In a Retrieval-Augmented Generation system, you may want:

| Task | Model |
|---|---|
| Query rewriting | Small fast model |
| Answer generation | Stronger model |
| Summarization | Lightweight model |
| Evaluation | Separate judge model |

Llama-Swap lets these models sit behind one API endpoint.

## AI Agents

AI agents may need different models for different roles:

| Agent Role | Model Type |
|---|---|
| Planner | Strong reasoning model |
| Tool caller | Fast instruction model |
| Summarizer | Small model |
| Code assistant | Code-focused model |

## Privacy-Focused Workflows

Local models are useful when data should not leave your machine.

Examples include private notes, internal company documents, academic work, sensitive logs, and offline experiments.

## Cost Control

Cloud LLM APIs charge per token. Local models do not charge per request after setup.

You still pay through hardware, electricity, and time, but you avoid per-call API pricing.


# 23. Common Errors and Fixes

## Error 1: `llama-server: command not found`

Meaning: `llama.cpp` is not installed or not on your PATH.

Fix:

```bash
brew install llama.cpp
which llama-server
```

## Error 2: model file not found

Meaning: your config path is wrong.

Fix:

```bash
ls -lh llama-swap-local-llms/models
```

Then update the model path in `config.yaml`.

## Error 3: port already in use

Meaning: another app is already using the port.

Fix:

Change:

```bash
--listen 127.0.0.1:8080
```

to:

```bash
--listen 127.0.0.1:8090
```

## Error 4: model is too slow

Possible reasons:

- model is too large
- CPU-only inference
- not enough RAM
- too many tokens requested

Fix:

- use smaller models
- reduce `max_tokens`
- use quantized GGUF files
- use GPU acceleration if available

## Error 5: empty or strange response

Possible reasons:

- wrong endpoint
- wrong prompt format
- model not instruction-tuned
- chat template mismatch

Fix:

- try `/v1/chat/completions`
- use an instruct model
- keep prompts simple
- test with a smaller prompt first


# 24. Best Practices

## Start small

Do not begin with a 7B or 13B model if you are learning.

Start with 135M, 500M, 1B, or 1.5B models, then move upward.

## Keep model names simple

Good:

```yaml
"qwen2.5"
"smollm2"
"mistral"
"coder"
```

Bad:

```yaml
"Qwen-2.5-0.5B-Instruct-Q4_K_M-final-latest-test"
```

## Use absolute paths in config

Absolute paths reduce confusion.

Example:

```text
/Users/yourname/llama-swap-local-llms/models/model.gguf
```

## Do not commit model files to GitHub

Model files can be very large.

Add this to `.gitignore`:

```text
models/
*.gguf
```

## Document your experiments

For each model, document:

- model name
- size
- quantization
- speed
- memory usage
- quality
- best use case


# 25. Recommended GitHub Folder Structure

For your GitHub repository, I recommend:

```text
Applied-Data-Science-and-AI-Lab/
│
├── llms-and-rag/
│   └── local-llm-infrastructure/
│       └── llama-swap-multiple-local-llms/
│           ├── llama_swap_multiple_local_llms.ipynb
│           ├── README.md
│           ├── config.example.yaml
│           ├── .gitignore
│           └── images/
```

Do **not** upload downloaded GGUF model files to GitHub. They are large and should be downloaded by users.


# 26. Example `.gitignore`

Create a `.gitignore` file:

```text
# Local model files
models/
*.gguf

# Archives
*.tar.gz
*.zip

# Python
__pycache__/
.ipynb_checkpoints/

# Environment files
.env
.venv/
venv/

# OS files
.DS_Store
```


# 27. Example README for This Notebook

You can use this inside the folder:

```markdown
# Running Multiple Local LLMs with Llama-Swap

This project demonstrates how to run and switch between multiple local LLMs on a single machine using Llama-Swap and llama.cpp.

## What it covers

- Local LLM serving
- GGUF model files
- llama.cpp and llama-server
- Llama-Swap configuration
- OpenAI-compatible API calls
- Model switching
- Python client examples
- Troubleshooting

## Models used

- SmolLM2-135M-Instruct-GGUF
- Qwen2.5-0.5B-Instruct-GGUF

## Why this matters

This setup helps reduce API costs, improve privacy, and build better local AI infrastructure for RAG systems, AI agents, and model experimentation.
```


# 28. Full End-to-End Command Summary

Here is the complete terminal workflow.

```bash
# 1. Create project folder
mkdir -p llama-swap-local-llms/models
cd llama-swap-local-llms

# 2. Install llama.cpp
brew install llama.cpp

# 3. Install Hugging Face CLI
pip install -U "huggingface_hub[cli]"

# 4. Install jq
brew install jq

# 5. Download models
hf download bartowski/SmolLM2-135M-Instruct-GGUF \
  SmolLM2-135M-Instruct-Q4_K_M.gguf \
  --local-dir models

hf download bartowski/Qwen2.5-0.5B-Instruct-GGUF \
  Qwen2.5-0.5B-Instruct-Q4_K_M.gguf \
  --local-dir models

# 6. Download Llama-Swap from GitHub releases manually
# Choose the correct file for your OS and CPU.
# Extract it:
tar -xzf llama-swap.tar.gz
chmod +x llama-swap

# 7. Create config.yaml
# Add your models and llama-server commands.

# 8. Start Llama-Swap
./llama-swap --config config.yaml --listen 127.0.0.1:8080

# 9. Test a model
curl -s http://localhost:8080/v1/chat/completions \
  -H "Content-Type: application/json" \
  -H "Authorization: Bearer no-key" \
  -d '{
        "model": "qwen2.5",
        "messages": [
          {"role": "user", "content": "Explain local LLMs simply."}
        ],
        "max_tokens": 100
      }' | jq
```


# 29. What You Should Study After This

After understanding this notebook, study these topics next:

1. GGUF quantization
2. llama.cpp performance tuning
3. OpenAI-compatible APIs
4. RAG architecture
5. Vector databases
6. LangChain with local models
7. FastAPI wrapper around local LLMs
8. Model evaluation
9. Agentic AI routing
10. Local deployment and monitoring

This setup is a strong foundation for practical AI engineering.


# 30. Final Takeaway

Most people think local LLMs are only about downloading a model and chatting with it.

But real AI engineering is about:

- routing
- serving
- switching
- reliability
- cost control
- privacy
- integration with applications

Llama-Swap is useful because it turns multiple local LLMs into one manageable API endpoint.

That makes it much easier to build local AI assistants, RAG systems, model comparison tools, AI agents, and private AI workflows.

This is a small but important step toward understanding real-world AI infrastructure.


# 31. References

Use these official resources to stay updated:

- Llama-Swap GitHub repository: https://github.com/mostlygeek/llama-swap
- Llama-Swap configuration docs: https://github.com/mostlygeek/llama-swap/blob/main/docs/configuration.md
- llama.cpp server docs: https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md
- Hugging Face CLI docs: https://huggingface.co/docs/huggingface_hub/en/guides/cli
- Hugging Face download docs: https://huggingface.co/docs/huggingface_hub/en/guides/download
